In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
DATA_DIR = Path("output")
JOBLIB_PATH = DATA_DIR / "modelo_priorizacao_alertas.joblib"

df_sicar = pd.read_pickle(DATA_DIR / "treinamento_modelo_sicar_ac.pkl")
df_mapbiomas = pd.read_pickle(DATA_DIR / "mapbiomas_alerts_model_ac_bkp.pkl")

df_sicar.shape, df_mapbiomas.shape, list(df_mapbiomas.columns)

((62491, 19),
 (689, 26),
 ['alerts',
  'property_propertyCode',
  'property_carType',
  'property_areaHa',
  'property_state',
  'property_stateAcronym',
  'property_version',
  'property_carUpdatedAt',
  'property_insertedAt',
  'property_boundingBox',
  'alerts_alertCode',
  'alerts_areaHa',
  'alerts_detectedAt',
  'alerts_publishedAt',
  'alerts_sources',
  'alerts_statusName',
  'alerts_boundingBox_neLat',
  'alerts_boundingBox_neLng',
  'alerts_boundingBox_swLat',
  'alerts_boundingBox_swLng',
  'alerts_coordenates_latitude',
  'alerts_coordenates_longitude',
  'alerts_imageAcquiredBeforeAt',
  'alerts_imageAcquiredAfterAt',
  'alerts_publishedImages',
  'property'])

In [3]:
df = df_mapbiomas.merge(
    df_sicar,
    how="inner",
    left_on=["property_propertyCode"],
    right_on=["cod_imovel"],
)
print(f"linhas no join: {len(df)} | imóveis: {df['property_propertyCode'].nunique()}")
df["property_propertyCode"].value_counts()

linhas no join: 639 | imóveis: 500


property_propertyCode
AC-1200401-91E88A0EADFD4B62B5051A51A7B41D3E    15
AC-1200609-6DB9F5FE5E4A4049840B7FAB7F7805E7     6
AC-1200609-53BF522DEDC64DB39254401B64DBD417     6
AC-1200500-40A0478118384FFAAD6B75DC63C9ECCF     6
AC-1200609-E2C13DE1391B4831AEAA4D2A045A40D7     6
                                               ..
AC-1200013-B9E0F5B6EB6D49F387DA1ACD92125E01     1
AC-1200013-0E340120DB5840B5870D33D942A57324     1
AC-1200013-875D77F7CCC84816B84CE7E223F6773E     1
AC-1200013-1B8462C3FDF7429AA85590ED20BBA737     1
AC-1200609-244357B72D544FFEB46B4A4C4E3A0A47     1
Name: count, Length: 500, dtype: int64

In [4]:
DROP_COLS = {
    "geometry",
    "total_features",
    "number_matched",
    "number_returned",
    "start_index",
    "page",
    "remaining",
    "is_pagination",
}


def first_existing(frame: pd.DataFrame, names: list[str], default=None):
    for name in names:
        if name in frame.columns:
            return frame[name]
    if default is None:
        return pd.Series(np.nan, index=frame.index)
    return pd.Series(default, index=frame.index)


def to_datetime_series(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", utc=True)


def sources_as_text(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "desconhecido"
    if isinstance(value, (list, tuple, set)):
        return "|".join(sorted(str(item) for item in value)) or "desconhecido"
    return str(value)


work = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore").copy()

work["area_alerta_ha"] = pd.to_numeric(
    first_existing(work, ["alerts_areaHa", "areaHa", "alert_areaHa"]),
    errors="coerce",
)
work["area_imovel_ha"] = pd.to_numeric(
    first_existing(work, ["area", "property_areaHa"]),
    errors="coerce",
)
work["modulos_fiscais"] = pd.to_numeric(first_existing(work, ["m_fiscal"]), errors="coerce")
work["fracao_imovel"] = work["area_alerta_ha"] / work["area_imovel_ha"].replace(0, np.nan)

detected = to_datetime_series(
    first_existing(work, ["alerts_detectedAt", "detectedAt", "alerts_publishedAt", "publishedAt"])
)
ref_date = detected.max()
work["recencia_dias"] = (ref_date - detected).dt.total_seconds() / 86400.0

work["n_alertas_imovel"] = work.groupby("property_propertyCode")["property_propertyCode"].transform(
    "size"
)
work["status_alerta"] = first_existing(work, ["alerts_statusName", "statusName"], default="desconhecido").astype(str)
work["fonte_alerta"] = first_existing(work, ["alerts_sources", "sources"]).map(sources_as_text)
work["status_imovel"] = first_existing(work, ["status_imovel"], default="desconhecido").astype(str)
work["condicao"] = first_existing(work, ["condicao"], default="desconhecido").astype(str)
work["tipo_imovel"] = first_existing(work, ["tipo_imovel"], default="desconhecido").astype(str)
work["uf"] = first_existing(work, ["uf", "property_stateAcronym"], default="desconhecido").astype(str).str.upper()

work[
    [
        "property_propertyCode",
        "area_alerta_ha",
        "area_imovel_ha",
        "fracao_imovel",
        "recencia_dias",
        "n_alertas_imovel",
        "status_imovel",
        "condicao",
    ]
].head()

,property_propertyCode,area_alerta_ha,area_imovel_ha,fracao_imovel,recencia_dias,n_alertas_imovel,status_imovel,condicao
0,AC-1200708-19C3C6A0A7B6488096185809637AC4AF,NaN,935.0576,NaN,NaN,1,PE,"Analisado, aguardando atendimento a notificação"
1,AC-1200179-CF396FF9F78E4A0E8CCA9CF80716144A,6.9998,295.3080,0.023703,974.0,1,AT,"Analisado, aguardando atendimento a notificação"
2,AC-1200138-218DCFB19D064EF38E435A4281FCEBB4,NaN,2.9585,NaN,NaN,1,AT,Aguardando análise
3,AC-1200500-FB62BF39455A442591A7D60DBCFA800C,NaN,4528.6282,NaN,NaN,1,CA,Cancelado por decisão administrativa
4,AC-1200401-9B67C502E4A945F9B30766960853306C,4.2876,1750.4796,0.002449,2038.0,6,AT,"Analisado, aguardando atendimento a notificação"


In [5]:
def minmax(series: pd.Series) -> pd.Series:
    s = series.astype(float)
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)


condicao_lower = work["condicao"].str.lower()
peso_car = np.where(condicao_lower.str.contains("notifica", na=False), 1.0, 0.0)
peso_car = np.where(condicao_lower.str.contains("aguardando análise", na=False), 0.6, peso_car)
peso_car = np.where(work["status_imovel"].str.upper().isin(["PE", "CA", "RE"]), np.maximum(peso_car, 0.7), peso_car)

work["score_prioridade"] = (
    0.30 * minmax(work["area_alerta_ha"].fillna(0))
    + 0.25 * minmax(work["fracao_imovel"].fillna(0))
    + 0.20 * (1 - minmax(work["recencia_dias"].fillna(work["recencia_dias"].median())))
    + 0.15 * minmax(work["n_alertas_imovel"].fillna(1))
    + 0.10 * peso_car
)

n_bins = min(3, work["score_prioridade"].nunique())
label_order = ["baixa", "media", "alta"][:n_bins]
work["prioridade"] = pd.qcut(
    work["score_prioridade"].rank(method="first"),
    q=n_bins,
    labels=label_order,
)

work["prioridade"].value_counts().sort_index()

prioridade
baixa    213
media    213
alta     213
Name: count, dtype: int64

In [6]:
NUMERIC_FEATURES = [
    "area_alerta_ha",
    "area_imovel_ha",
    "modulos_fiscais",
    "fracao_imovel",
    "recencia_dias",
    "n_alertas_imovel",
]
CATEGORICAL_FEATURES = [
    "status_alerta",
    "fonte_alerta",
    "status_imovel",
    "condicao",
    "tipo_imovel",
    "uf",
]

X = work[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = work["prioridade"].astype(str)

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            NUMERIC_FEATURES,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                    ),
                ]
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])

test_size = 0.3 if len(X) >= 10 else 0.25
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

labels = [lab for lab in ["baixa", "media", "alta"] if lab in set(y)]
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))
print("matriz de confusão\n", confusion_matrix(y_test, y_pred, labels=labels))
print("F1 macro (indicativo, amostra exploratória):", round(f1_macro, 3))

              precision    recall  f1-score   support

       baixa       0.87      0.86      0.87        64
       media       0.73      0.75      0.74        64
        alta       0.83      0.81      0.82        64

    accuracy                           0.81       192
   macro avg       0.81      0.81      0.81       192
weighted avg       0.81      0.81      0.81       192

matriz de confusão
 [[55  7  2]
 [ 7 48  9]
 [ 1 11 52]]
F1 macro (indicativo, amostra exploratória): 0.808


In [7]:
ohe = pipeline.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
cat_names = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
feature_names = NUMERIC_FEATURES + cat_names
importances = pipeline.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
)
importance_df

,feature,importance
26,"condicao_Analisado, aguardando atendimento a n...",0.189307
29,condicao_Em análise,0.125602
1,area_imovel_ha,0.086542
6,status_alerta_nan,0.074527
4,recencia_dias,0.071487
21,fonte_alerta_desconhecido,0.068336
0,area_alerta_ha,0.059623
7,status_alerta_published,0.056222
2,modulos_fiscais,0.051544
3,fracao_imovel,0.047132


In [8]:
proba = pipeline.predict_proba(X)
classes = list(pipeline.classes_)
idx_alta = classes.index("alta") if "alta" in classes else int(np.argmax([c == classes[-1] for c in classes]))

artifact = {
    "pipeline": pipeline,
    "feature_names": NUMERIC_FEATURES + CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "label_order": labels,
    "classes_": classes,
    "metrics": {
        "f1_macro": float(f1_macro),
        "n_rows": int(len(X)),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "exploratory": True,
    },
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, JOBLIB_PATH)
JOBLIB_PATH.resolve()

PosixPath('/Users/pedrohenriqueklein/Documents/gestao/planejamento-gestao-projetos/notebooks/output/modelo_priorizacao_alertas.joblib')

In [9]:
loaded = joblib.load(JOBLIB_PATH)
pipe = loaded["pipeline"]
sample = X.iloc[[0]]
pred = pipe.predict(sample)[0]
proba_alta = float(pipe.predict_proba(sample)[0][list(pipe.classes_).index(pred)])
{
    "prioridade": pred,
    "confianca_classe": proba_alta,
    "classes": list(pipe.classes_),
    "n_features": loaded["feature_names"],
}

{'prioridade': 'media',
 'confianca_classe': 0.6501697629295558,
 'classes': ['alta', 'baixa', 'media'],
 'n_features': ['area_alerta_ha',
  'area_imovel_ha',
  'modulos_fiscais',
  'fracao_imovel',
  'recencia_dias',
  'n_alertas_imovel',
  'status_alerta',
  'fonte_alerta',
  'status_imovel',
  'condicao',
  'tipo_imovel',
  'uf']}